# The complete memory API

A tour of all three memory layers - short-term, long-term, and reasoning - plus `get_context`, which reads across them. Each section matches one lesson in the workshop.

Run the cells in order. Jumping in at a later section? Use **Run All Above** on its first cell - the reset in the setup keeps re-runs clean. Your agent's own `learner` session is never touched.

In [ ]:
# Setup: the client, connected to your own instance.
import logging
import os

from dotenv import load_dotenv
from neo4j_agent_memory import MemoryClient, MemorySettings
from neo4j_agent_memory.config import Neo4jConfig, EmbeddingConfig, ExtractionConfig, ExtractorType
from neo4j_agent_memory.schema.models import EntityRef

load_dotenv()

# Silence the driver's deprecation notices for the vector-index queries.
logging.getLogger("neo4j.notifications").setLevel(logging.ERROR)

settings = MemorySettings(
    neo4j=Neo4jConfig(
        uri=os.environ["NEO4J_URI"],
        username=os.environ["NEO4J_USERNAME"],
        password=os.environ["NEO4J_PASSWORD"],
    ),
    embedding=EmbeddingConfig(api_key=os.environ["OPENAI_API_KEY"]),
    extraction=ExtractionConfig(extractor_type=ExtractorType.LLM),
)

SESSION_ID = "learner-api-tour"
BULK_SESSION = "learner-api-tour-bulk"  # a throwaway session for the batch/extract demos

memory = MemoryClient(settings)
await memory.connect()
print("connected")

In [ ]:
# Reset only this tour's own data, so your agent's real "learner" session
# survives. Entities carry no session, so reach them through the tour's
# messages - which is why this runs before clear_session removes them.
tour_sessions = [SESSION_ID, BULK_SESSION]

# Entities the tour extracted, minus any another session also mentions.
await memory.graph.execute_write(
    """
    MATCH (c:Conversation)-[:HAS_MESSAGE]->(:Message)-[:MENTIONS]->(e:Entity)
    WHERE c.session_id IN $sessions
      AND NOT EXISTS {
        MATCH (oc:Conversation)-[:HAS_MESSAGE]->(:Message)-[:MENTIONS]->(e)
        WHERE NOT oc.session_id IN $sessions
      }
    DETACH DELETE e
    """,
    {"sessions": tour_sessions},
)

# Reasoning traces carry session_id, so they scope directly; their steps
# and tool calls go with them.
await memory.graph.execute_write(
    """
    MATCH (rt:ReasoningTrace)-[:HAS_STEP|USES_TOOL*0..]->(n)
    WHERE rt.session_id IN $sessions
    DETACH DELETE n
    """,
    {"sessions": tour_sessions},
)

# Sweep entities nothing mentions any more - the dedup demo's misspelling,
# and any stranded by a message wipe. Extraction cannot re-link an entity
# that already exists, so remove strays and let it recreate them fresh.
await memory.graph.execute_write(
    """
    MATCH (e:Entity)
    WHERE NOT (e:__KGBuilder__ OR e:__Entity__)
      AND NOT EXISTS { (:Message)-[:MENTIONS]->(e) }
    DETACH DELETE e
    """,
    {},
)

# Clear the tour's conversations and their messages.
await memory.short_term.clear_session(SESSION_ID)
await memory.short_term.clear_session(BULK_SESSION)
# Facts and Preferences are global in this package - they carry no session
# to scope on - so they are left untouched, to protect your agent's own.
print("tour sessions reset")

## Short-term memory

Store a conversation, then read it back three ways: in order, by meaning, and as a summary.

In [ ]:
await memory.short_term.add_message(SESSION_ID, "user", "How could GraphRAG help me explore the complete works of Shakespeare?")
await memory.short_term.add_message(SESSION_ID, "assistant", "GraphRAG would connect the characters and places across Shakespeare's plays, like Hamlet and Verona, so you can traverse between them.")
conversation = await memory.short_term.get_conversation(SESSION_ID)
hits = await memory.short_term.search_messages(query="using GraphRAG to explore Shakespeare", session_id=SESSION_ID, limit=10)
summary = await memory.short_term.get_conversation_summary(SESSION_ID)
print("messages:", len(conversation.messages), "| top hit:", hits[0].content[:45] if hits else None, "| topics:", summary.key_topics)

`add_messages_batch` loads a whole transcript in one call, into a throwaway session - and skips extraction by default.

In [ ]:
await memory.short_term.add_messages_batch(BULK_SESSION, [
    {"role": "user", "content": "I'm reading Pride and Prejudice by Jane Austen."},
    {"role": "assistant", "content": "A classic. Jane Austen set much of it in Hertfordshire, in England."},
    {"role": "user", "content": "Who are the main characters?"},
    {"role": "assistant", "content": "Elizabeth Bennet and Mr Darcy are at the heart of it, alongside Elizabeth's sister Jane."},
    {"role": "user", "content": "Where does Mr Darcy live?"},
    {"role": "assistant", "content": "At Pemberley, his estate in Derbyshire."},
    {"role": "user", "content": "And when was it published?"},
    {"role": "assistant", "content": "Jane Austen published it in 1813."},
])
bulk = await memory.short_term.get_conversation(BULK_SESSION)
print("bulk messages loaded:", len(bulk.messages))

In [ ]:
# Managing the conversation: list every stored session.
all_sessions = await memory.short_term.list_sessions()
print("sessions:", [s.session_id for s in all_sessions])

## Long-term memory: entities and relationships

The batch load skipped extraction, so the bulk session has no entities yet. `extract_entities_from_session` runs the extractor over a stored session on demand.

In [ ]:
stats = await memory.short_term.extract_entities_from_session(BULK_SESSION)
print("extracted from bulk session:", stats)

Extraction created `Shakespeare` without an embedding; `add_entity` merges onto it and embeds it, so `search_entities` can find it by meaning.

In [ ]:
await memory.long_term.add_entity(name="Shakespeare", entity_type="PERSON", description="English playwright")
entities = await memory.long_term.search_entities(query="Who is the most famous Elizabethan poet and playwright?")
print("search_entities found:", entities[0].name if entities else None)

In [ ]:
# Relationships: connect two entities that already exist.
shakespeare = await memory.long_term.get_entity_by_name("Shakespeare")
hamlet = await memory.long_term.get_entity_by_name("Hamlet")
await memory.long_term.add_relationship(shakespeare, hamlet, "WROTE")
print("recorded: Shakespeare WROTE Hamlet")

In [ ]:
# Ask the graph: search finds the entity by meaning, the graph supplies the connection.
topics = await memory.long_term.search_entities(query="Who wrote the play about the Prince of Denmark?")
related = await memory.long_term.get_related_entities(topics[0].id) if topics else []
print("ask the graph:", topics[0].name if topics else None, "->", [entity.name for entity, relationship in related])

## Deduplication

Add a near-duplicate - it is auto-flagged with a `SAME_AS` edge - then find the flagged pair and settle it.

In [ ]:
await memory.long_term.add_entity(name="Shakespere", entity_type="PERSON")
print("added the misspelling")

In [ ]:
pairs = await memory.long_term.find_potential_duplicates()
print("flagged pairs:", [(a.name, b.name) for a, b, score in pairs])

In [ ]:
duplicate = await memory.long_term.get_entity_by_name("Shakespere")
canonical = await memory.long_term.get_entity_by_name("Shakespeare")
await memory.long_term.review_duplicate(duplicate.id, canonical.id, confirm=True)
print("merged:", duplicate.name, "->", canonical.name)

## Facts and preferences

Both are deliberate writes: a fact is a standalone subject-predicate-object claim; a preference hangs off its `:User`.

In [ ]:
await memory.long_term.add_fact(subject="learner", predicate="struggles_with", obj="GraphRAG")
facts = await memory.long_term.search_facts(query="what does the learner struggle with")
print("search_facts found:", facts[0].as_triple if facts else None)

In [ ]:
pref = await memory.long_term.add_preference(category="learning_style", preference="Learns best from worked examples", user_identifier="learner")
prefs = await memory.long_term.search_preferences(query="how does the learner like to study")
print("search_preferences found:", prefs[0].preference if prefs else None)

In [ ]:
# When a preference changes, supersede it - the history stays intact.
new_pref = await memory.long_term.add_preference(category="learning_style", preference="Prefers video walkthroughs", user_identifier="learner")
await memory.long_term.supersede_preference(pref.id, new_pref.id)
print("superseded:", pref.preference, "->", new_pref.preference)

## Reasoning memory

Record a decision as a trace, wired into the conversation and the graph. Uses the conversation stored in the short-term section - if you jumped straight here, Run All Above first.

In [ ]:
user_message_id = conversation.messages[0].id
trace = await memory.reasoning.start_trace(
    session_id=SESSION_ID,
    task="Recommend the next topic for the learner",
    triggered_by_message_id=user_message_id,
    user_identifier="learner",
)
print("trace open:", trace.task)

In [ ]:
step = await memory.reasoning.add_step(
    trace_id=trace.id,
    thought="Read what the learner struggles with",
    action="search_facts",
    observation="The learner struggles with GraphRAG",
)
await memory.reasoning.record_tool_call(
    step_id=step.id,
    tool_name="search_facts",
    arguments={"query": "struggles_with"},
    message_id=user_message_id,
    touched_entities=[EntityRef(name="GraphRAG")],
)
print("step recorded:", step.action)

In [ ]:
await memory.reasoning.complete_trace(trace.id, outcome="Recommended the GraphRAG lesson", success=True)
print("trace closed")

## Querying traces

Find past traces by task similarity, then read the best match back in full.

In [ ]:
similar = await memory.reasoning.get_similar_traces(task="Recommend a topic for the learner to study next", limit=3)
print("similar past traces:")
for past in similar:
    print(f"- {past.task} -> {past.outcome} (success: {past.success})")

full = await memory.reasoning.get_trace_with_steps(similar[0].id) if similar else None
print("full trace, step by step:")
for s in full.steps if full else []:
    print(f"- thought: {s.thought} | action: {s.action} | observation: {s.observation}")

## Combined context

`get_context` reads across all three layers and assembles one block of text, sized for an LLM prompt.

In [ ]:
context = await memory.get_context(query="What should this learner focus on?", session_id=SESSION_ID)
print(context)

Done. To start the tour fresh, run again from the top - the reset keeps it clean. Close the client when you are finished with the notebook.

In [ ]:
await memory.close()
print("closed")